<a href="https://colab.research.google.com/github/Rajapaksha-RMDD/pusl3123-ai-ml-coursework/blob/main/PUSL3123_Exercise_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Regularisation and Generalisation (CIFAR-10 CNN)




In [ ]:
import time, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------- Configuration ----------------
SPLIT_SEED   = 42          # same split as Exercise 2
SEEDS        = [42, 43, 44]  # repeat every experiment with these seeds (use [42] for a quick run)
MAIN_SEED    = 42          # seed used for the learning-curve plots
EPOCHS       = 10          # same as Exercises 1 and 2
BATCH_SIZE   = 64
LR           = 1e-3
DROPOUT_P    = 0.5
ROTATION_DEG = 10          # small rotations
TRANSLATE    = 0.1         # translations up to 10% of image size
NUM_WORKERS  = 2           # set to 0 if DataLoader workers cause problems on your machine

## Data


In [ ]:
normalize = transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))

eval_transform = transforms.Compose([transforms.ToTensor(), normalize])

aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=ROTATION_DEG, translate=(TRANSLATE, TRANSLATE)),
    transforms.ToTensor(),
    normalize,
])

full_clean = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=eval_transform)
full_aug   = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=aug_transform)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=eval_transform)

# Same call as Exercise 2 -> same indices
train_part, val_part = random_split(
    full_clean, [45000, 5000],
    generator=torch.Generator().manual_seed(SPLIT_SEED)
)
train_idx, val_idx = train_part.indices, val_part.indices
assert len(set(train_idx) & set(val_idx)) == 0, "train/validation overlap!"

train_clean = Subset(full_clean, train_idx)   # clean training images
train_aug   = Subset(full_aug,   train_idx)   # same images, random augmentation on the fly
val_clean   = Subset(full_clean, val_idx)     # validation is NEVER augmented

print("Train:", len(train_clean), "| Validation:", len(val_clean), "| Test:", len(test_dataset))

classes = full_clean.classes

def make_loader(dataset, shuffle, seed=0):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                      generator=g if shuffle else None)

test_loader = make_loader(test_dataset, shuffle=False)

### Check: what do the augmented training images look like

In [ ]:
def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for j, idx in enumerate(train_idx[:8]):
    img_c, lab = full_clean[idx]
    img_a, _   = full_aug[idx]
    axes[0, j].imshow(denorm(img_c)); axes[0, j].set_title(classes[lab], fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(denorm(img_a)); axes[1, j].axis("off")
axes[0, 0].text(-0.15, 0.5, "Original", transform=axes[0, 0].transAxes, rotation=90, va="center", ha="right")
axes[1, 0].text(-0.15, 0.5, "Augmented", transform=axes[1, 0].transAxes, rotation=90, va="center", ha="right")
plt.suptitle("Training images: original vs. augmented")
plt.tight_layout()
plt.savefig("ex3_augmentation_examples.png", dpi=150)
plt.show()

##Model


In [ ]:
class CNN(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)    # trainable layer 1
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)   # trainable layer 2
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity()
        self.fc1   = nn.Linear(64 * 8 * 8, 10)                     # trainable layer 3

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.dropout(x)          # active only in model.train()
        return self.fc1(x)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("Baseline params:", count_params(CNN(0.0)), "| With dropout params:", count_params(CNN(DROPOUT_P)))
print(CNN(DROPOUT_P))

## Training and evaluation utilities


In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        total_loss += criterion(outputs, labels).item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct / total      # per-sample loss, accuracy %

def run_experiment(name, dropout_p, augment, seed, epochs=EPOCHS, verbose=True):
    set_seed(seed)                                   # same seed -> same initial weights for every config
    model = CNN(dropout_p).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    train_loader      = make_loader(train_aug if augment else train_clean, shuffle=True, seed=seed)
    train_eval_loader = make_loader(train_clean, shuffle=False)
    val_loader        = make_loader(val_clean, shuffle=False)

    hist = {"train_loss": [], "val_loss": [], "train_acc_running": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * labels.size(0)
            correct  += (outputs.argmax(1) == labels).sum().item()
            total    += labels.size(0)
        val_loss, val_acc = evaluate(model, val_loader)
        hist["train_loss"].append(run_loss / total)
        hist["train_acc_running"].append(100.0 * correct / total)
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)
        if verbose:
            print(f"  [{name} | seed {seed}] epoch {epoch+1:2d}/{epochs} "
                  f"train loss {hist['train_loss'][-1]:.4f} acc {hist['train_acc_running'][-1]:.2f}% | "
                  f"val loss {val_loss:.4f} acc {val_acc:.2f}%")

    # Final evaluation (after training is finished): clean train accuracy, validation, and ONE test evaluation
    _, train_acc_clean = evaluate(model, train_eval_loader)
    val_loss, val_acc  = evaluate(model, val_loader)
    test_loss, test_acc = evaluate(model, test_loader)

    result = {
        "Configuration": name, "Seed": seed, "Params": count_params(model),
        "Train Acc (clean)": train_acc_clean, "Val Acc": val_acc, "Test Acc": test_acc,
        "Gap (Train-Test)": train_acc_clean - test_acc,
        "Final Train Loss (running)": hist["train_loss"][-1],
        "Final Val Loss": val_loss, "Test Loss": test_loss,
    }
    return result, hist

## Run the experiments


In [ ]:
CONFIGS = {
    "Baseline":                (0.0,       False),
    f"Dropout (p={DROPOUT_P})": (DROPOUT_P, False),
    "Augmentation":            (0.0,       True),
    "Dropout + Augmentation":  (DROPOUT_P, True),
}

results, histories = [], {}
t0 = time.time()
for seed in SEEDS:
    for name, (p, aug) in CONFIGS.items():
        res, hist = run_experiment(name, p, aug, seed, verbose=(seed == MAIN_SEED))
        results.append(res); histories[(name, seed)] = hist
        print(f"seed {seed} | {name:26s} | train(clean) {res['Train Acc (clean)']:.2f}% | "
              f"test {res['Test Acc']:.2f}% | gap {res['Gap (Train-Test)']:.2f} pts | "
              f"elapsed {(time.time()-t0)/60:.1f} min")

results_df = pd.DataFrame(results)
results_df.to_csv("ex3_all_runs.csv", index=False)
results_df

##Results table

In [ ]:
order = list(CONFIGS.keys())
metrics = ["Train Acc (clean)", "Val Acc", "Test Acc", "Gap (Train-Test)", "Test Loss"]

agg = results_df.groupby("Configuration")[metrics].agg(["mean", "std"]).reindex(order)
summary = pd.DataFrame(index=order)
for m in metrics:
    summary[m] = [f"{agg[(m,'mean')][c]:.2f} ± {agg[(m,'std')][c]:.2f}" if m != "Test Loss"
                  else f"{agg[(m,'mean')][c]:.3f} ± {agg[(m,'std')][c]:.3f}" for c in order]
summary["Params"] = results_df.groupby("Configuration")["Params"].first().reindex(order)
print(f"Mean ± std over seeds {SEEDS} (accuracies in %, gap in percentage points)")
display(summary)

print(f"Single run (seed {MAIN_SEED}):")
display(results_df[results_df.Seed == MAIN_SEED].set_index("Configuration").reindex(order)[
    ["Train Acc (clean)", "Val Acc", "Test Acc", "Gap (Train-Test)", "Test Loss"]].round(2))

summary.to_csv("ex3_summary_table.csv")

##Figures

In [ ]:
# Figure 1: training vs validation loss for each configuration (main seed)
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharey=True)
for ax, name in zip(axes.ravel(), order):
    h = histories[(name, MAIN_SEED)]
    ep = range(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], marker="o", label="Training loss (running)")
    ax.plot(ep, h["val_loss"],   marker="o", label="Validation loss")
    ax.set_title(name); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(True); ax.legend()
plt.suptitle(f"Training vs validation loss (seed {MAIN_SEED})")
plt.tight_layout()
plt.savefig("ex3_loss_curves.png", dpi=150)
plt.show()

# Figure 2: validation loss and accuracy of all configurations overlaid
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name in order:
    h = histories[(name, MAIN_SEED)]
    ep = range(1, len(h["val_loss"]) + 1)
    axes[0].plot(ep, h["val_loss"], marker="o", label=name)
    axes[1].plot(ep, h["val_acc"],  marker="o", label=name)
axes[0].set_title("Validation loss"); axes[1].set_title("Validation accuracy (%)")
for ax in axes: ax.set_xlabel("Epoch"); ax.grid(True); ax.legend()
plt.tight_layout()
plt.savefig("ex3_validation_curves.png", dpi=150)
plt.show()

# Figure 3: train vs test accuracy and generalisation gap (mean ± std over seeds)
g = results_df.groupby("Configuration")
tr_m, tr_s = g["Train Acc (clean)"].mean().reindex(order), g["Train Acc (clean)"].std().reindex(order).fillna(0)
te_m, te_s = g["Test Acc"].mean().reindex(order),          g["Test Acc"].std().reindex(order).fillna(0)
gp_m, gp_s = g["Gap (Train-Test)"].mean().reindex(order),  g["Gap (Train-Test)"].std().reindex(order).fillna(0)

x = np.arange(len(order)); w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
b1 = axes[0].bar(x - w/2, tr_m, w, yerr=tr_s, capsize=4, label="Train accuracy (clean)")
b2 = axes[0].bar(x + w/2, te_m, w, yerr=te_s, capsize=4, label="Test accuracy")
axes[0].bar_label(b1, fmt="%.1f", padding=2); axes[0].bar_label(b2, fmt="%.1f", padding=2)
axes[0].set_xticks(x); axes[0].set_xticklabels(order, rotation=15); axes[0].set_ylabel("Accuracy (%)")
axes[0].set_ylim(50, 100); axes[0].set_title("Training vs test accuracy"); axes[0].legend(); axes[0].grid(axis="y", alpha=.3)

b3 = axes[1].bar(x, gp_m, 0.5, yerr=gp_s, capsize=4, color="tab:red")
axes[1].bar_label(b3, fmt="%.1f", padding=2)
axes[1].set_xticks(x); axes[1].set_xticklabels(order, rotation=15)
axes[1].set_ylabel("Train − test accuracy (percentage points)")
axes[1].set_title("Generalisation gap (lower = less overfitting)"); axes[1].grid(axis="y", alpha=.3)
plt.suptitle(f"Mean ± std over seeds {SEEDS}")
plt.tight_layout()
plt.savefig("ex3_comparison.png", dpi=150)
plt.show()

## Data driven summary


In [ ]:
base = results_df[results_df.Configuration == "Baseline"]
base_test, base_gap = base["Test Acc"].mean(), base["Gap (Train-Test)"].mean()
base_std = base["Test Acc"].std() if len(base) > 1 else float("nan")

print(f"Baseline: test {base_test:.2f}%, gap {base_gap:.2f} pts (seed std of test acc: {base_std:.2f})\n")
for name in order[1:]:
    r = results_df[results_df.Configuration == name]
    d_test = r["Test Acc"].mean() - base_test
    d_gap  = r["Gap (Train-Test)"].mean() - base_gap
    noise  = max(base_std, r["Test Acc"].std()) if len(r) > 1 else float("nan")
    if np.isnan(noise):
        verdict = "(single seed – cannot judge significance)"
    elif abs(d_test) > noise:
        verdict = "larger than seed noise"
    else:
        verdict = "within seed noise -> treat as inconclusive"
    print(f"{name:26s}: test acc {d_test:+.2f} pts ({verdict}); gap {d_gap:+.2f} pts "
          f"({'smaller' if d_gap < 0 else 'larger'} gap)")

## does 10 epochs favour the baseline?


In [ ]:
RUN_LONG, LONG_EPOCHS = False, 30

if RUN_LONG:
    long_results, long_hist = [], {}
    for name, (p, aug) in CONFIGS.items():
        res, hist = run_experiment(name, p, aug, MAIN_SEED, epochs=LONG_EPOCHS, verbose=False)
        long_results.append(res); long_hist[name] = hist
    long_df = pd.DataFrame(long_results).set_index("Configuration")
    display(long_df[["Train Acc (clean)", "Val Acc", "Test Acc", "Gap (Train-Test)"]].round(2))
    plt.figure(figsize=(8, 4.5))
    for name in order:
        plt.plot(range(1, LONG_EPOCHS + 1), long_hist[name]["val_loss"], label=name)
    plt.xlabel("Epoch"); plt.ylabel("Validation loss"); plt.title(f"{LONG_EPOCHS}-epoch validation loss")
    plt.grid(True); plt.legend(); plt.tight_layout(); plt.savefig("ex3_long_training.png", dpi=150); plt.show()